# Notebook 10.4 — Streamlit App Integration & Testing

## Objective
I validate the final local application path: Streamlit UI → hybrid query engine → Ollama/Qwen3 4B → evidence display. I also check that Phase 9 assets and dependencies are available before launching the app.

## Notebook presentation rule
Relevant smoke tests are followed by concise **Result & conclusion** cells.

In [8]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path
import importlib.util

ROOT = Path("..").resolve()
APP_DIR = ROOT / "app"

def _find_phase9():
    env = __import__("os").getenv("RETAINIQ_PHASE9_DIR")
    if env:
        return Path(env)
    for name in ["Phase 9 — GenAI  RAG Knowledge Layer", "Phase 9 — GenAI RAG Knowledge Layer", "Phase 9 - GenAI  RAG Knowledge Layer", "Phase 9 - GenAI RAG Knowledge Layer"]:
        p = ROOT.parent / name
        if p.exists():
            return p
    for p in ROOT.parent.glob("*Phase 9*"):
        if p.is_dir():
            return p
    return ROOT.parent / "Phase 9 — GenAI  RAG Knowledge Layer"

PHASE9_DIR = _find_phase9()

required = [
    PHASE9_DIR / "outputs/retrieval_chunks.csv",
    PHASE9_DIR / "outputs/retrieval_vectors.npy",
    PHASE9_DIR / "outputs/retrieval_manifest.json",
]
missing = [str(p) for p in required if not p.exists()]

print("Phase 9 directory:", PHASE9_DIR)
print("Missing required assets:", missing)
assert not missing, "Phase 9 retrieval assets are missing. Set RETAINIQ_PHASE9_DIR or complete Phase 9 first."

print(f"Result: All {len(required)} required Phase 9 retrieval assets are available.")

def print_result(message):
    print(f"Result: {message}")


Phase 9 directory: C:\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\Phase 9 — GenAI  RAG Knowledge Layer
Missing required assets: []
Result: All 3 required Phase 9 retrieval assets are available.


### Result & conclusion
The application depends on the saved Phase 9 retrieval index. I verify those files before testing the app so missing upstream assets fail early and clearly.

In [10]:
dependencies = ['requests','pandas','numpy','streamlit','sentence_transformers','mysql.connector']
status = {}
for name in dependencies:
    module_name = 'sentence_transformers' if name == 'sentence_transformers' else ('mysql.connector' if name == 'mysql.connector' else name)
    status[name] = importlib.util.find_spec(module_name) is not None

print(pd.Series(status, name='available'))
print_result(f"{sum(status.values())}/{len(status)} required Python dependencies are discoverable in the current kernel environment.")

requests                 True
pandas                   True
numpy                    True
streamlit                True
sentence_transformers    True
mysql.connector          True
Name: available, dtype: bool
Result: 6/6 required Python dependencies are discoverable in the current kernel environment.


### Result & conclusion
This is an environment smoke test. A missing dependency should be fixed in the active environment before the Streamlit app is launched.

In [11]:
sys.path.insert(0, str(APP_DIR))
from query_engine import answer_query, PARAMETERS

print("Configured model:", PARAMETERS.get("llm_model"))
print("Configured endpoint:", PARAMETERS.get("llm_base_url"))
print("LLM enabled:", PARAMETERS.get("llm_enabled"))

result = answer_query("What does a 20 percent recovery scenario mean?", mysql_password=None, top_k=5)
print("Mode:", result["mode"])
print("Answer preview:\n", result["answer"][:1500])
print("\nEvidence IDs:", result["retrieved"]["evidence_id"].tolist())


Configured model: qwen3:4b
Configured endpoint: http://localhost:11434/v1
LLM enabled: true
Mode: llm
Answer preview:
 A "20 percent recovery scenario" means a scenario where the recovery rate is 20%. According to evidence [2], this is one of the default scenarios used in Phase 7, and it represents the percentage of revenue at risk that could be recovered through retention actions. The formula for estimated recovered revenue in this scenario is `Estimated Recovered Revenue = Revenue at Risk × 0.20`. Evidence [2] explicitly states that these default scenarios (5%, 10%, 20%, and 30%) are "scenario assumptions, not predictions."

Evidence IDs: ['[1]', '[2]']


### Result & conclusion
The backend smoke test verifies that the same query engine used by the app can retrieve evidence and attempt local Qwen3 generation. The reported mode tells me whether the run used the LLM or a fallback.

In [12]:
app_path = APP_DIR / 'streamlit_app.py'
assert app_path.exists(), f'Missing app file: {app_path}'
content = app_path.read_text(encoding='utf-8')
required_markers = ['st.chat_input', 'st.chat_message', 'Evidence used', 'answer_query']
missing_markers = [m for m in required_markers if m not in content]
print('Missing app markers:', missing_markers)
assert not missing_markers, 'The Streamlit app is missing one or more expected UI/backend integration markers.'
print_result('The Streamlit app contains the expected chat input, message history, evidence display, and backend integration hooks.')

Missing app markers: []
Result: The Streamlit app contains the expected chat input, message history, evidence display, and backend integration hooks.


In [ ]:
# Rank the customer segments by revenue at risk.
# cd ".\Phase 10 — AI Assistant & Business Q&A"
# streamlit run app\streamlit_app.py

### Result & conclusion
The app smoke test confirms the intended interface elements are wired into the backend. This does not replace manual browser testing, so I still need to launch Streamlit and ask real questions.

## Launch command
From the Phase 10 project root, run:

```bash
streamlit run app/streamlit_app.py
```

Keep Ollama available locally with `qwen3:4b` installed. The app is designed to show the retrieved evidence under each answer.

## Phase 10 completion criteria
I consider Phase 10 complete when: (1) the app launches, (2) a normal RetainIQ question returns `mode = llm`, (3) evidence is visible under the answer, (4) comparison questions do not overclaim when evidence is incomplete, and (5) the app remains usable when the LLM is unavailable by showing transparent fallback evidence.